# EdNet 유저 로그 불러오기

유저별로 CSV가 나뉘어 있어, **유저 5명 표본**을 하나로 합쳐 구조를 파악합니다.

## 1. KT1 유저 5명 합치기
- `u[1-5].csv`: 파일 5개를 한 번에 읽음
- `filename = true`: 유저 ID 컬럼이 없어서 출처 파일로 유저를 구분

In [2]:
import duckdb

con = duckdb.connect("../data/ednet.duckdb")

df = con.sql("""
SELECT *
FROM read_csv('../data/KT1/u[1-5].csv', filename = true)
""").df()

df

,timestamp,solving_id,question_id,user_answer,elapsed_time,filename
0,1565096190868,1,q5012,b,38000,..\data\KT1\u1.csv
1,1565096221062,2,q4706,c,24000,..\data\KT1\u1.csv
2,1565096293432,3,q4366,b,68000,..\data\KT1\u1.csv
3,1565096339668,4,q4829,a,42000,..\data\KT1\u1.csv
4,1565096401774,5,q6528,b,59000,..\data\KT1\u1.csv
...,...,...,...,...,...,...
6122,1567603866226,811,q621,a,11000,..\data\KT1\u5.csv
6123,1567603892727,812,q1030,b,18000,..\data\KT1\u5.csv
6124,1567604449086,813,q956,a,22000,..\data\KT1\u5.csv
6125,1567604487238,814,q692,a,18000,..\data\KT1\u5.csv


## 2. 함수화: `load_users(level, pattern)`
KT2~KT4에도 같은 작업을 반복하므로, 데이터 폴더(`level`)와 유저 패턴(`pattern`)만 인자로 뺐습니다.

In [3]:
import duckdb

con = duckdb.connect("../data/ednet.duckdb")

def load_users(level, pattern="u[1-5]"):
    path = f"../data/{level}/{pattern}.csv"
    return con.sql(f"""
    SELECT *
    FROM read_csv('{path}', filename = true)
    """).df()

## 3. KT2 불러오기
KT1이 문제 단위 기록이라면, KT2는 `enter` → `respond` → `submit` 처럼 **행동(action) 단위** 기록입니다.

In [4]:
df2 = load_users("KT2")
df2

,timestamp,action_type,item_id,source,user_answer,platform,filename
0,1565096151269,enter,b3544,diagnosis,NaN,mobile,..\data\KT2\u1.csv
1,1565096187972,respond,q5012,diagnosis,b,mobile,..\data\KT2\u1.csv
2,1565096194904,submit,b3544,diagnosis,NaN,mobile,..\data\KT2\u1.csv
3,1565096195001,enter,b3238,diagnosis,NaN,mobile,..\data\KT2\u1.csv
4,1565096218682,respond,q4706,diagnosis,c,mobile,..\data\KT2\u1.csv
...,...,...,...,...,...,...,...
4067,1567604489658,submit,b692,sprint,NaN,web,..\data\KT2\u5.csv
4068,1567604517322,enter,b565,sprint,NaN,web,..\data\KT2\u5.csv
4069,1567604528181,respond,q565,sprint,a,web,..\data\KT2\u5.csv
4070,1567604534421,respond,q565,sprint,b,web,..\data\KT2\u5.csv


## 4. KT3 불러오기
KT2와 컬럼은 같지만, 해설(`e`) 열람이나 `quit` 같은 행동까지 포함돼 기록이 더 많습니다 (4,072행 → 6,798행).

In [5]:
df3 = load_users("KT3")
df3

,timestamp,action_type,item_id,source,user_answer,platform,filename
0,1565096151269,enter,b3544,diagnosis,NaN,mobile,..\data\KT3\u1.csv
1,1565096187972,respond,q5012,diagnosis,b,mobile,..\data\KT3\u1.csv
2,1565096194904,submit,b3544,diagnosis,NaN,mobile,..\data\KT3\u1.csv
3,1565096195001,enter,b3238,diagnosis,NaN,mobile,..\data\KT3\u1.csv
4,1565096218682,respond,q4706,diagnosis,c,mobile,..\data\KT3\u1.csv
...,...,...,...,...,...,...,...
6793,1567604515500,quit,e692,sprint,NaN,web,..\data\KT3\u5.csv
6794,1567604517322,enter,b565,sprint,NaN,web,..\data\KT3\u5.csv
6795,1567604528181,respond,q565,sprint,a,web,..\data\KT3\u5.csv
6796,1567604534421,respond,q565,sprint,b,web,..\data\KT3\u5.csv


## 5. KT4 불러오기
`play_audio`, `pause_audio` 같은 세부 행동과 재생 위치(`cursor_time`) 컬럼이 추가돼 기록이 가장 많습니다 (6,798행 → 9,467행).

In [6]:
df4 = load_users("KT4")
df4

,timestamp,action_type,item_id,cursor_time,source,user_answer,platform,filename
0,1565096151269,enter,b3544,<NA>,diagnosis,NaN,mobile,..\data\KT4\u1.csv
1,1565096187972,respond,q5012,<NA>,diagnosis,b,mobile,..\data\KT4\u1.csv
2,1565096194904,submit,b3544,<NA>,diagnosis,NaN,mobile,..\data\KT4\u1.csv
3,1565096195001,enter,b3238,<NA>,diagnosis,NaN,mobile,..\data\KT4\u1.csv
4,1565096218682,respond,q4706,<NA>,diagnosis,c,mobile,..\data\KT4\u1.csv
...,...,...,...,...,...,...,...,...
9462,1567604517376,play_audio,b565,17767,sprint,NaN,web,..\data\KT4\u5.csv
9463,1567604528181,respond,q565,<NA>,sprint,a,web,..\data\KT4\u5.csv
9464,1567604533783,pause_audio,b565,15882,sprint,NaN,web,..\data\KT4\u5.csv
9465,1567604534421,respond,q565,<NA>,sprint,b,web,..\data\KT4\u5.csv
